# 08 · 端到端實戰：一份完整的分析報告

整條軌道的收尾。把**問題 → 資料 → 清理 → EDA → 特徵 → 模型 → 結論**走完整一輪,產出一份能交付的分析。這正是資料科學家的日常產出長相。

**問題:什麼樣的鐵達尼號乘客比較容易生還?能不能預測?**

## 1. 取得與清理

In [ ]:
%pip install -q -U seaborn scikit-learn

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"])
print(f"清理完成:{df.shape[0]} 位乘客,整體生還率 {df['survived'].mean():.1%}")

## 2. 關鍵發現(EDA 濃縮)

In [ ]:
print("生還率 by 性別:");  print(df.groupby("sex")["survived"].mean().round(2))
print("\n生還率 by 艙等:"); print(df.groupby("pclass")["survived"].mean().round(2))

## 3. 特徵工程 + 建模

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

d = df.copy()
d["sex"] = d["sex"].map({"male": 0, "female": 1})
d["family_size"] = d["sibsp"] + d["parch"] + 1
feats = ["pclass", "sex", "age", "fare", "family_size"]
X_train, X_test, y_train, y_test = train_test_split(
    d[feats], d["survived"], test_size=0.2, random_state=42, stratify=d["survived"]
)
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f"準確率:{accuracy_score(y_test, pred):.1%}\n")
print(classification_report(y_test, pred, target_names=["died", "survived"]))

## 4. 哪些因素最重要?

隨機森林的特徵重要度,把整份分析收斂成一句話。

In [ ]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
print("特徵重要度:"); print(importance.round(3))

## 5. 結論(這就是要交付的東西)

> **鐵達尼號的生還,主要由「性別、艙等、票價」決定**:女性與高艙等乘客生還率遠高於平均,印證了「婦孺與富人優先」。用這幾個特徵,隨機森林能以約八成準確率預測一位乘客是否生還。

一份好的資料分析,最後一定回到**一句人話的結論** + 支撐它的證據(圖表、檢定、模型)。

## 軌道小結

你走完了資料科學的完整流程:

- **流程與載入**(01)→ **清理**(02)→ **EDA**(03)→ **視覺化**(04)
- **特徵工程**(05)→ **統計檢定**(06)→ **建模**(07)→ **完整報告**(08)

**會清資料、會探索、會說故事、能接上模型**——這就是資料科學家把原始資料變成決策的核心能力。接下來想深入建模,`ml/scikit-learn` 在等你。📈